# TRINETRA Kaggle runner
Attach **two** inputs: `happyparsi/trinetra-dataset` and the code dataset (`TRINETRA/` folder from `trinetra_code.zip`).
Set `ACCOUNT` and `PHASE`, enable **GPU T4 x2** and **Internet**, then *Run All*.
Numbers produced here are OUR RESULTS on a prototype subset; never compare them directly with full-benchmark papers.

In [ ]:
ACCOUNT = "P"        # A = fire CNN + colour baseline + video | B = zero-shot | C = deepfake | D = EO heads + geo | F = fusion + RL (after A-C) | P = 7 physics models (tests + real-data hindcast, GLOF, RL env, Geo-PINN) | ALL = everything in one session
PHASE   = "phase2"   # "phase2" = fast (1 seed, few epochs) | "final" = 3 seeds, full epochs
DATASET = "/kaggle/input/trinetra-dataset"
READY   = "/kaggle/input/trinetra-ready-v1/npz"   # used if a teammate already published converted NPZs
GITHUB  = ""         # optional fallback: https URL of a repo that contains TRINETRA/

In [ ]:
import base64, io, zipfile, os
_Z = '...'
zipfile.ZipFile(io.BytesIO(base64.b64decode(_Z))).extractall('/kaggle/working')
print('embedded TRINETRA code unpacked:', os.path.isdir('/kaggle/working/TRINETRA/trinetra/kaggle'))

In [ ]:
import os, subprocess, shutil, glob, pathlib
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode: print(f"[exit {r.returncode}]")
    return r.returncode
# 1) locate code
hits = glob.glob("/kaggle/input/**/trinetra/kaggle/run_hackathon.sh", recursive=True)
CODE = "/kaggle/working/TRINETRA"
if os.path.isdir(CODE + "/trinetra/kaggle"):
    pass  # already unpacked (embedded package)
elif hits:
    src = str(pathlib.Path(hits[0]).parents[2]); shutil.copytree(src, CODE, dirs_exist_ok=True)
elif GITHUB:
    sh(f"git clone --depth 1 {GITHUB} /kaggle/working/repo && cp -r /kaggle/working/repo/TRINETRA {CODE}")
else:
    raise SystemExit("Attach the TRINETRA code dataset (or set GITHUB)")
print("code at", CODE)

In [ ]:
EPOCHS, SEEDS = ("5", "0") if PHASE == "phase2" else ("15", "0 1 2")
os.environ.update(OUT="/kaggle/working/out", EPOCHS=EPOCHS, SEEDS=SEEDS, PY="python",
                  KAGGLE_INPUT=DATASET, DFIRE_N="3000" if PHASE == "phase2" else "4000",
                  CIFAKE_N="5000" if PHASE == "phase2" else "10000")
sh("pip -q install timm tifffile imagecodecs h5py" + (" open_clip_torch" if ACCOUNT in ("B", "ALL") else ""))
sh("nvidia-smi --query-gpu=name,memory.total --format=csv || true")

In [ ]:
# 2) data: resolve the attached team dataset, discover the real folder roots, build only the NPZs this account needs.
# Why: happyparsi/trinetra-dataset is ONE bundle with human-readable folder titles (e.g. "CIFAKE/",
# "AI Generated Images vs Real Images/"), not the per-slug folders run_hackathon.sh defaults to, and on
# Python 3.13 os.walk / Path.rglob do not follow symlinked mounts. So we resolve real paths and walk with followlinks.
import json, re, sys
MARKERS = {"cifake", "ai generated images vs real images", "avenue_dataset", "crisismmd_v2.0"}

def _kids(d):
    try: return {e.lower() for e in os.listdir(d)}
    except OSError: return set()

def find_dataset_root():
    cands = [DATASET] + sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*")) \
          + sorted(glob.glob("/kaggle/input/*/*/*")) + sorted(glob.glob("/kaggle/input/*/*/*/*"))
    seen = []
    for c in cands:
        r = os.path.realpath(c)
        if os.path.isdir(r) and r not in seen:
            seen.append(r)
            if _kids(r) & MARKERS:
                return r
    # fallback: a directory literally named like the slug that has content
    for r in seen:
        if "trinetra-dataset" in r.lower() and _kids(r):
            return r
    print("Could not find the team dataset. Contents of /kaggle/input (3 levels):")
    sh("find -L /kaggle/input -maxdepth 3 -type d 2>/dev/null | head -80")
    raise SystemExit("DATASET NOT FOUND: right panel -> Add Input -> happyparsi/trinetra-dataset -> Add, then Run All again.")

DATASET = find_dataset_root()
os.environ["KAGGLE_INPUT"] = DATASET
print("dataset at", DATASET)
sh(f"ls -1 '{DATASET}' | head -80")

IMG = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
def walk_dirs(root, max_depth=7):
    """yield (dirpath, {lower_name: real_name} of subdirs, set of lower file names, n_images); follows symlinks."""
    base = root.rstrip("/").count("/")
    for cur, dirs, files in os.walk(root, followlinks=True):
        if cur.count("/") - base >= max_depth:
            dirs[:] = []
        yield cur, {d.lower(): d for d in dirs}, {f.lower(): f for f in files}, sum(f.lower().endswith(IMG) for f in files)

print("[discover] walking dataset (follows symlinks; ~1-3 min on 61 GB) ...")
found = {}
def put(key, val):
    found.setdefault(key, []).append(val)
for cur, sub, files, nimg in walk_dirs(DATASET):
    low = cur.lower()
    if "data.yaml" in files:
        put("HOMEFIRE" if "home" in low else "DFIRE", cur)
    if {"images", "labels"} <= sub.keys() and ("fire" in low or "smoke" in low):
        put("HOMEFIRE_NOYAML" if "home" in low else "DFIRE_NOYAML", cur)
    if os.path.basename(cur).lower() == "training" and {"fire", "nofire"} <= sub.keys():
        put("DEEPFIRE", (os.path.dirname(cur), f"{os.path.basename(cur)}/{sub['fire']}", f"{os.path.basename(cur)}/{sub['nofire']}"))
    if {"fire_images", "non_fire_images"} <= sub.keys() or {"fire_images", "non-fire_images"} <= sub.keys():
        put("FIREDS", (cur, sub["fire_images"], sub.get("non_fire_images") or sub.get("non-fire_images")))
    w = [sub[k] for k in ("fogsmog", "sandstorm", "lightning", "rain") if k in sub]
    if len(w) >= 2:
        put("WEATHER", (cur, w))
    s = [sub[k] for k in ("street", "buildings", "forest", "mountain") if k in sub]
    if len(s) >= 3:
        put("SCENES", (cur, s))
    if {"real", "fake"} <= sub.keys():
        face_like = any(t in low for t in ("final dataset", "face", "stylegan"))
        put("FACES" if face_like else "CIFAKE", (cur, sub["real"], sub["fake"]))
    if {"aiartdata", "realart"} <= sub.keys():
        put("AIART", (cur, sub["aiartdata"], sub["realart"]))
    if {"fire", "smoke"} <= sub.keys():
        kinds = [{k.lower(): k for k in (os.listdir(os.path.join(cur, sub[h])) if os.path.isdir(os.path.join(cur, sub[h])) else [])}
                 for h in ("fire", "smoke")]
        if all({"pos", "neg"} <= k.keys() for k in kinds):
            put("FIRESENSE", (cur, [f"{sub[h]}/{k[p]}={lab}" for h, k in zip(("fire", "smoke"), kinds)
                                    for p, lab in (("pos", h), ("neg", "none"))]))
    if any(f.startswith("image_") and f.endswith(".h5") for f in files):
        put("L4S", os.path.dirname(os.path.dirname(cur)) if os.path.basename(cur).lower() == "img" else cur)
    if any("s1hand" in f and f.endswith((".tif", ".tiff")) for f in files):
        put("S1F", cur)
    if "hephaestus" in low and any(f.endswith(".json") for f in files):
        put("HEPH_ANN", cur)
    if "hephaestus" in low and nimg:
        put("HEPH", cur)
    for f in files:
        if f.endswith(".csv"):
            if "landslide" in f: put("GLC_CSV", os.path.join(cur, files[f]))
            if "glof" in f: put("GLOF_CSV", os.path.join(cur, files[f]))
            if "imerg" in f or "cloudburst" in f: put("IMERG_CSV", os.path.join(cur, files[f]))
    if os.path.basename(cur).lower() == "images" and os.path.basename(os.path.dirname(cur)).lower() == "train" \
            and ("flame" in low or "candle" in low) and nimg:
        gp = os.path.dirname(os.path.dirname(cur))
        put("CANDLE", (gp, os.path.relpath(cur, gp)))

def first(key, prefer=None):
    v = found.get(key, [])
    if prefer:
        v = sorted(v, key=lambda x: (0 if re.search(prefer, str(x).lower()) else 1, len(str(x))))
    else:
        v = sorted(v, key=lambda x: len(str(x)))
    return v[0] if v else None

roots = {
    "DFIRE": first("DFIRE", r"d-?fire|smoke.?fire|fire.?smoke"),
    "DFIRE_NOYAML": first("DFIRE_NOYAML", r"d-?fire|smoke.?fire|fire.?smoke"),
    "HOMEFIRE": first("HOMEFIRE") or first("HOMEFIRE_NOYAML"),
    "DEEPFIRE": first("DEEPFIRE", r"forest"),
    "FIREDS": first("FIREDS"),
    "WEATHER": first("WEATHER"),
    "SCENES": first("SCENES", r"train"),
    "CANDLE": first("CANDLE"),
    "FIRESENSE": first("FIRESENSE"),
    "CIFAKE": first("CIFAKE", r"cifake"),
    "FACES": first("FACES", r"final dataset"),
    "AIART": first("AIART"),
    "L4S": first("L4S", r"landslide4sense|l4s"),
    "S1F": first("S1F", r"sen1floods11"),
    "HEPH": first("HEPH"),
    "HEPH_ANN": first("HEPH_ANN", r"annotation"),
    "GLC_CSV": first("GLC_CSV", r"global.?landslide|glc"),
    "GLOF_CSV": first("GLOF_CSV", r"hmaglof"),
    "IMERG_CSV": first("IMERG_CSV"),
}
# CIFAKE ships train/{REAL,FAKE} and test/{REAL,FAKE}: use the common parent so both splits are read
if roots["CIFAKE"]:
    c = [x for x in found["CIFAKE"] if "cifake" in x[0].lower()] or found["CIFAKE"]
    roots["CIFAKE"] = (os.path.commonpath([x[0] for x in c]) if len(c) > 1 else c[0][0], c[0][1], c[0][2])
print("[discover]", json.dumps(roots, indent=1, default=str))
os.makedirs("/kaggle/working/trinetra_stage", exist_ok=True)
json.dump({"dataset": DATASET, "selected": roots, "all_candidates": found},
          open("/kaggle/working/trinetra_stage/discovered_roots.json", "w"), indent=1, default=str)

# build_npz.py / build_eo.py use Path.rglob; on Python >= 3.13 make it follow symlinked mounts too
RUNNER = "/kaggle/working/adapter_runner.py"
open(RUNNER, "w").write(
    "import pathlib, runpy, sys\n"
    "if sys.version_info >= (3, 13):\n"
    "    _r = pathlib.Path.rglob\n"
    "    pathlib.Path.rglob = lambda self, pat, **k: _r(self, pat, recurse_symlinks=True, **k)\n"
    "S = sys.argv[1]\n"
    "sys.path.insert(0, str(pathlib.Path(S).parent)); sys.argv = [S] + sys.argv[2:]\n"
    "runpy.run_path(S, run_name='__main__')\n")
BNPZ, BEO = f"{CODE}/data_adapters/build_npz.py", f"{CODE}/data_adapters/build_eo.py"
NPZ = "/kaggle/working/npz"; MAN = "/kaggle/working/out/manifests"
os.makedirs(NPZ, exist_ok=True); os.makedirs(MAN, exist_ok=True)
def q(x): return "'" + str(x).replace("'", "'\\''") + "'"
def npz(args, label, script=None):
    script = script or BNPZ
    rc = sh(f"python {RUNNER} {script} {args}")
    if rc: print(f"[warn] {label} skipped (exit {rc})")
    return rc == 0
def heldout(raw, final):
    if not os.path.exists(raw):
        print(f"[warn] {raw} missing; {final} not built"); return False
    if os.path.exists(f"{NPZ}/dfire_all.npz"):
        return npz(f"dedupe --train {NPZ}/dfire_all.npz --test {raw} --out {final}", f"dedupe {final}")
    shutil.copy(raw, final); print(f"[warn] {final} NOT deduped (no D-Fire)"); return True

def build_fire_sets():
    S = os.environ.get("SIZE", "128"); N = os.environ.get("DFIRE_N", "3000")
    # main fire set (D-Fire YOLO). Classes from data.yaml; if it is missing, D-Fire order is 0=smoke 1=fire.
    if roots["DFIRE"] or roots["DFIRE_NOYAML"]:
        r, cls = (roots["DFIRE"], "auto") if roots["DFIRE"] else (os.path.dirname(roots["DFIRE_NOYAML"]), "0=smoke 1=fire")
        if npz(f"yolo --root {q(r)} --classes {cls} --seq-bucket 50 --task fire --size {S} --max-images {N} "
               f"--out {NPZ}/dfire_all.npz --manifest {MAN}/dfire_seed0.csv", "D-Fire"):
            npz(f"subset --npz {NPZ}/dfire_all.npz --keep train val --out {NPZ}/fire.npz", "fire subset")
            npz(f"subset --npz {NPZ}/dfire_all.npz --keep test --out {NPZ}/fire_test_same_source.npz", "fire test subset")
    if roots["DEEPFIRE"]:
        r, f, nf = roots["DEEPFIRE"]
        if npz(f"folder --root {q(r)} --map {q(f + '=fire')} {q(nf + '=none')} --task fire --size {S} --max-per-class 760 "
               f"--out {NPZ}/deepfire_raw.npz --manifest {MAN}/deepfire_seed0.csv", "DeepFire"):
            heldout(f"{NPZ}/deepfire_raw.npz", f"{NPZ}/deepfire.npz")
    if roots["FIREDS"]:
        r, f, nf = roots["FIREDS"]
        if npz(f"folder --root {q(r)} --map {q(f + '=fire')} {q(nf + '=none')} --task fire --size {S} --max-per-class 755 "
               f"--out {NPZ}/firedataset_raw.npz", "FIRE Dataset"):
            heldout(f"{NPZ}/firedataset_raw.npz", f"{NPZ}/heldout_firedataset.npz")
    if roots["HOMEFIRE"]:
        h = roots["HOMEFIRE"]; cls = "auto" if h in found.get("HOMEFIRE", []) else "0=fire 1=smoke"
        npz(f"yolo --root {q(h)} --classes {cls} --task fire --size {S} --max-images 1500 --out {NPZ}/heldout_homefire.npz", "home-fire")
    if roots["WEATHER"]:
        r, ks = roots["WEATHER"]
        npz(f"folder --root {q(r)} --map {' '.join(q(k + '=none') for k in ks)} --task fire --size {S} --max-per-class 250 "
            f"--out {NPZ}/stress_weather.npz", "weather")
    if roots["CANDLE"]:
        r, k = roots["CANDLE"]
        npz(f"folder --root {q(r)} --map {q(k + '=none')} --seq-bucket 50 --task fire --size {S} --max-per-class 600 "
            f"--out {NPZ}/stress_small_flames.npz", "candle set")
    if roots["SCENES"]:
        r, ks = roots["SCENES"]
        npz(f"folder --root {q(r)} --map {' '.join(q(k + '=none') for k in ks)} --task fire --size {S} --max-per-class 200 "
            f"--out {NPZ}/stress_scenes.npz", "intel scenes")

def build_video_set():
    if roots["FIRESENSE"]:
        r, maps = roots["FIRESENSE"]
        npz(f"video --root {q(r)} --map {' '.join(q(m) for m in maps)} --clip 1 --every 10 --size {os.environ.get('SIZE', '128')} "
            f"--out {NPZ}/firesense_frames.npz", "FireSense")
    else:
        print("[warn] FireSense (fire/{pos,neg}, smoke/{pos,neg} videos) not found; video stage will be skipped")

def build_media_sets():
    n = os.environ.get("CIFAKE_N", "10000")
    for key, out, cap, extra in (("CIFAKE", "media", n, ""), ("FACES", "faces_test", "1000", ""), ("AIART", "aiart_test", "540", "")):
        if not roots[key]:
            print(f"[warn] {key} not found; {out}.npz skipped"); continue
        r, a, b = roots[key]
        if key == "AIART":   # (root, AiArtData, RealArt)
            m = f"{q(a + '=fake')} {q(b + '=real')}"
        else:                # (root, REAL, FAKE)
            m = f"{q(a + '=real')} {q(b + '=fake')}"
        npz(f"folder --root {q(r)} --map {m} --task deepfake --size 32 --max-per-class {cap} --out {NPZ}/{out}.npz", key)

def build_eo_sets():
    z = os.environ.get("EO_SIZE", "64")
    if roots["L4S"]:
        npz(f"l4s --root {q(roots['L4S'])} --out {NPZ}/l4s.npz --size {z} --max-n {os.environ.get('EO_L4S_N', '512')}", "L4S", BEO)
    if roots["S1F"]:
        npz(f"sen1floods11 --root {q(roots['S1F'])} --out {NPZ}/s1f.npz --size {z} --max-n {os.environ.get('EO_S1_N', '400')} "
            f"--holdout-event Bolivia", "Sen1Floods11", BEO)
    if roots["HEPH"] and roots["HEPH_ANN"]:
        npz(f"hephaestus --root {q(roots['HEPH'])} --annotations {q(roots['HEPH_ANN'])} --out {NPZ}/heph.npz --size {z} "
            f"--max-n {os.environ.get('EO_HEPH_N', '512')}", "Hephaestus", BEO)
    elif roots["HEPH"]:
        print("[warn] Hephaestus images found but annotations missing; skipped to avoid frameID leakage")
    if not (roots["L4S"] or roots["S1F"] or roots["HEPH"]):
        print("[warn] no EO source (Landslide4Sense image_*.h5 / Sen1Floods11 *S1Hand*.tif / Hephaestus) in the dataset; "
              "attach tekbahadurkshetri/landslide4sense or smabrarrajin/sen1floods11-essentials for eo-train")

# geo / physics stages read these CSVs from the environment (they otherwise download from NASA / GitHub)
for k in ("GLC_CSV", "GLOF_CSV", "IMERG_CSV"):
    if roots[k]: os.environ[k] = roots[k]; print(f"[env] {k}={roots[k]}")

STAGES = {"A": ["fire", "video"], "B": ["zeroshot"], "C": ["deepfake"], "D": ["eo-train", "geo"], "F": ["fuse"], "P": ["physics"],
          "ALL": ["fire", "deepfake", "zeroshot", "fuse", "video", "eo-train", "physics"]}[ACCOUNT]
os.environ["DATA"] = NPZ
if os.path.isdir(READY) and glob.glob(READY + "/*.npz"):
    os.environ["DATA"] = READY; print("using teammate's ready pack", READY)
else:
    if {"fire", "zeroshot", "video"} & set(STAGES): build_fire_sets()
    if "video" in STAGES: build_video_set()
    if "deepfake" in STAGES: build_media_sets()
    if "eo-train" in STAGES:
        sh("pip -q install h5py tifffile imagecodecs pillow"); build_eo_sets()
    if set(STAGES) <= {"physics", "geo", "fuse"}: print("no image conversion needed for", STAGES)

if glob.glob(os.environ["DATA"] + "/*.npz"):
    sh(f"python {CODE}/data_adapters/validate_ready_pack.py --data $DATA --report /kaggle/working/trinetra_stage/ready_pack_report.json")
sh("ls -la $DATA")
NEEDS = {"fire": ["fire.npz"], "zeroshot": ["fire.npz"], "video": ["firesense_frames.npz"], "deepfake": ["media.npz"],
         "eo-train": ["l4s.npz", "s1f.npz", "heph.npz"]}
missing = [st for st in STAGES if st in NEEDS and not any(os.path.exists(f"{os.environ['DATA']}/{f}") for f in NEEDS[st])]
if missing:
    print(f"[!] stages {missing} have no input NPZ and will be skipped. Check the [warn] lines above and "
          "/kaggle/working/trinetra_stage/discovered_roots.json.")
if missing and len(missing) == len([st for st in STAGES if st in NEEDS]) and not set(STAGES) - set(NEEDS):
    raise SystemExit("No input data could be built for this account; see the [warn] lines above.")

In [ ]:
# 3) train / evaluate for this account; results are collected after every stage so a timeout keeps partial outputs
R = f"bash {CODE}/trinetra/kaggle/run_hackathon.sh"
plans = {"A": ["fire", "video"], "B": ["zeroshot"], "C": ["deepfake"], "D": ["eo-train", "geo"], "F": ["fuse"], "P": ["physics"],
         "ALL": ["fire", "deepfake", "zeroshot", "fuse", "video", "eo-train", "physics"]}
status = {}
for stage in plans[ACCOUNT]:
    need = NEEDS.get(stage)
    if need and not any(os.path.exists(f"{os.environ['DATA']}/{f}") for f in need):
        print(f"[skip] {stage}: none of {need} in {os.environ['DATA']} (see the data cell above)"); status[stage] = "skipped: no input"
        continue
    if stage == "fuse" and not os.path.exists("/kaggle/working/out/fire/fire_s0_oof.npz"):
        print("[skip] fuse: needs /kaggle/working/out/fire/fire_s0_oof.npz from the fire stage"); status[stage] = "skipped: no fire OOF"
        continue
    status[stage] = sh(f"set -o pipefail; {R} {stage} 2>&1 | tee /kaggle/working/{stage}.log")
    sh(f"python {CODE}/trinetra/kaggle/collect_results.py --inp /kaggle/working/out --out /kaggle/working/out/results > /dev/null 2>&1")
print("stage exit codes:", status)
open("/kaggle/working/stage_status.json", "w").write(json.dumps(status, indent=2))

In [ ]:
# 4) collect + package (download out.zip or publish /kaggle/working/out as a private dataset version)
sh(f"python {CODE}/trinetra/kaggle/collect_results.py --inp /kaggle/working/out --out /kaggle/working/out/results")
sh("cp /kaggle/working/*.log /kaggle/working/stage_status.json /kaggle/working/out/ 2>/dev/null; cp -r /kaggle/working/trinetra_stage/*.json /kaggle/working/out/ 2>/dev/null")
shutil.make_archive(f"/kaggle/working/out_{ACCOUNT}_{PHASE}", "zip", "/kaggle/working/out")
sh("cat /kaggle/working/out/results/RESULTS.md 2>/dev/null | head -80")
if not any(v == 0 for v in status.values()):
    print("\n[!] No stage finished successfully, so RESULTS.md has no rows. Fix the first [warn]/[skip] above and Run All again.")
print("download:", f"/kaggle/working/out_{ACCOUNT}_{PHASE}.zip")